In [1]:
https://colab.research.google.com/github/GoogleCloudPlatform/generative-ai/blob/main/agents/gemini_data_analytics/intro_gemini_data_analytics_http.ipynb

SyntaxError: invalid syntax (1630052668.py, line 1)

In [5]:
!pip install google-auth

In [9]:
import json
import json as json_lib

import altair as alt
import pandas as pd
import requests
from IPython.display import HTML, display
from google import auth
from pygments import formatters, highlight, lexers
import textwrap

In [11]:
google.auth.default()

access_token = !gcloud auth application-default print-access-token
headers = {
    "Authorization": f"Bearer {access_token[0]}",
    "Content-Type": "application/json",
}

In [12]:
billing_project = "llm-studies"  # @param {type:"string"}

location = "global"
api_version = "v1beta"

system_instruction = "Think like an Analyst"  # @param {type:"string"}

bigquery_data_sources = {}
looker_data_sources = {}
looker_studio_data_sources = {}

In [13]:
# fmt: off
environment = "prod"  # @param ["prod", "staging", "autopush"]
# fmt: on

base_url = "https://geminidataanalytics.googleapis.com"

if environment == "autopush":
    base_url = "https://autopush-geminidataanalytics.sandbox.googleapis.com"
elif environment == "staging":
    base_url = "https://staging-geminidataanalytics.sandbox.googleapis.com"
else:
    base_url = "https://geminidataanalytics.googleapis.com"

In [14]:
base_url

'https://geminidataanalytics.googleapis.com'

In [16]:
bigquery_data_sources = {
    "bq": {
        "tableReferences": [
            {
                "projectId": "bigquery-public-data",
                "datasetId": "faa",
                "tableId": "us_airports",
            },
            {
                "projectId": "bigquery-public-data",
                "datasetId": "san_francisco",
                "tableId": "street_trees",
            },
            # Add more table references here
        ]
    }
}

# optional example queries (only leveraged for BigQuery datasources currently)
example_queries = [
    {
        "naturalLanguageQuestion": "What is the highest observed positive longitude?",
        "sqlQuery": "SELECT MAX(longitude) FROM airports",
    },
]

glossary_terms = [
    {
        "display_name": "Airport Code",
        "description": "A unique identifier for an airport, represented by a 3 character long identifier (the IATA code). E.g. 'JFK' for New York.",
        "labels": ["code", "IATA", "identifier"]
    },
]

use_glossary_terms = True # @param {type: "boolean"}
use_example_queries = True  # @param {type:"boolean"}

In [18]:
selected_datasource = "bigquery_data_sources"  # @param ["bigquery_data_sources", "looker_data_sources", "looker_studio_data_sources"]

datasource_map = {
    "bigquery_data_sources": bigquery_data_sources,
    "looker_data_sources": looker_data_sources,
    "looker_studio_data_sources": looker_studio_data_sources,
}

datasource_references = datasource_map[selected_datasource]

bq_selected = False
looker_selected = False
looker_studio_selected = False

if selected_datasource == "looker_data_sources":
  looker_selected = True
elif selected_datasource == "bigquery_data_sources":
  bq_selected = True
elif selected_datasource == "looker_studio_data_sources":
  looker_studio_selected = True

In [19]:
data_agent_id = "data_agent_1"  # @param {type:"string"}
data_agent_url = f"{base_url}/{api_version}/projects/{billing_project}/locations/{location}/dataAgents"
data_agent_payload = {
    "name": f"projects/{billing_project}/locations/{location}/dataAgents/{data_agent_id}",  # Optional
    "description": "This is the description of data_agent.",  # Optional
    "data_analytics_agent": {
        "published_context": {
            "datasource_references": datasource_references,
            "system_instruction": system_instruction,
            "options": {
                "analysis": {
                    # Optional - if wanting to use advanced analysis with python.
                    # Default is False.
                    "python": {"enabled": False}
                }
            },
        }
    },
}

if bq_selected:
    if use_example_queries:
        data_agent_payload["data_analytics_agent"]["published_context"]["example_queries"] = example_queries
    if use_glossary_terms:
        data_agent_payload["data_analytics_agent"]["published_context"]["glossary_terms"] = glossary_terms
elif looker_selected and use_looker_golden_queries:
    data_agent_payload["data_analytics_agent"]["published_context"]["looker_golden_queries"] = looker_golden_queries

params = {"data_agent_id": data_agent_id}  # Optional

data_agent_response = requests.post(
    data_agent_url, params=params, json=data_agent_payload, headers=headers
)

if data_agent_response.status_code == 200:
    print("Data Agent created successfully!")
    print(json.dumps(data_agent_response.json(), indent=2))
else:
    print(f"Error creating Data Agent: {data_agent_response.status_code}")
    print(data_agent_response.text)

Data Agent created successfully!
{
  "name": "projects/llm-studies/locations/global/operations/operation-1773073143615-64c99c2e2d1a3-6579c043-803608cb",
  "metadata": {
    "@type": "type.googleapis.com/google.cloud.geminidataanalytics.v1beta.OperationMetadata",
    "createTime": "2026-03-09T16:19:05.381688392Z",
    "target": "projects/llm-studies/locations/global/dataAgents/data_agent_1",
    "verb": "create",
    "requestedCancellation": false,
    "apiVersion": "v1beta"
  },
  "done": false
}


In [20]:
data_agent_id = "data_agent_1"  # @param {type:"string"}
data_agent_url = f"{base_url}/{api_version}/projects/{billing_project}/locations/{location}/dataAgents/{data_agent_id}"
data_agent_response = requests.get(data_agent_url, headers=headers)
if data_agent_response.status_code == 200:
    print("Fetched Data Agent successfully!")
    print(json.dumps(data_agent_response.json(), indent=2))
else:
    print(f"Error: {data_agent_response.status_code}")
    print(data_agent_response.text)

Fetched Data Agent successfully!
{
  "name": "projects/llm-studies/locations/global/dataAgents/data_agent_1",
  "description": "This is the description of data_agent.",
  "createTime": "2026-03-09T16:19:05.196095537Z",
  "updateTime": "2026-03-09T16:19:06.054657166Z",
  "dataAnalyticsAgent": {
    "publishedContext": {
      "systemInstruction": "Think like an Analyst",
      "options": {
        "analysis": {
          "python": {}
        }
      },
      "exampleQueries": [
        {
          "naturalLanguageQuestion": "What is the highest observed positive longitude?",
          "sqlQuery": "SELECT MAX(longitude) FROM airports"
        }
      ],
      "datasourceReferences": {
        "bq": {
          "tableReferences": [
            {
              "projectId": "bigquery-public-data",
              "datasetId": "faa",
              "tableId": "us_airports"
            },
            {
              "projectId": "bigquery-public-data",
              "datasetId": "san_francisco",

In [21]:
data_agent_url = f"{base_url}/{api_version}/projects/{billing_project}/locations/{location}/dataAgents"

data_agent_response = requests.get(data_agent_url, headers=headers)

if data_agent_response.status_code == 200:
    print("Data Agents Listed successfully!")
    print(json.dumps(data_agent_response.json(), indent=2))
else:
    print(f"Error Listing Data Agents: {data_agent_response.status_code}")
    print(data_agent_response.text)

Data Agents Listed successfully!
{
  "dataAgents": [
    {
      "name": "projects/llm-studies/locations/global/dataAgents/data_agent_1",
      "description": "This is the description of data_agent.",
      "createTime": "2026-03-09T16:19:05.196095537Z",
      "updateTime": "2026-03-09T16:19:06.054657166Z",
      "dataAnalyticsAgent": {
        "publishedContext": {
          "systemInstruction": "Think like an Analyst",
          "options": {
            "analysis": {
              "python": {}
            }
          },
          "exampleQueries": [
            {
              "naturalLanguageQuestion": "What is the highest observed positive longitude?",
              "sqlQuery": "SELECT MAX(longitude) FROM airports"
            }
          ],
          "datasourceReferences": {
            "bq": {
              "tableReferences": [
                {
                  "projectId": "bigquery-public-data",
                  "datasetId": "faa",
                  "tableId": "us_airports"

In [22]:
data_agent_id = "data_agent_1"  # @param {type:"string"}
conversation_id = "conversation_1"  # @param {type:"string"}
conversation_url = f"{base_url}/{api_version}/projects/{billing_project}/locations/{location}/conversations"
conversation_payload = {
    "agents": [
        f"projects/{billing_project}/locations/{location}/dataAgents/{data_agent_id}"
    ],
    "name": f"projects/{billing_project}/locations/{location}/conversations/{conversation_id}",
}
params = {"conversation_id": conversation_id}

conversation_response = requests.post(
    conversation_url, headers=headers, params=params, json=conversation_payload
)

if conversation_response.status_code == 200:
    print("Conversation created successfully!")
    print(json.dumps(conversation_response.json(), indent=2))
else:
    print(f"Error creating Conversation: {conversation_response.status_code}")
    print(conversation_response.text)

Conversation created successfully!
{
  "name": "projects/llm-studies/locations/global/conversations/conversation_1",
  "agents": [
    "projects/llm-studies/locations/global/dataAgents/data_agent_1"
  ],
  "createTime": "2026-03-09T16:25:46.528536Z",
  "lastUsedTime": "2026-03-09T16:25:47.051255Z"
}


In [23]:
conversation_id = "conversation_1"  # @param {type:"string"}
conversation_url = f"{base_url}/{api_version}/projects/{billing_project}/locations/{location}/conversations/{conversation_id}"
conversation_response = requests.get(conversation_url, headers=headers)
# Handle the response
if conversation_response.status_code == 200:
    print("Conversation fetched successfully!")
    print(json.dumps(conversation_response.json(), indent=2))
else:
    print(f"Error while fetching conversation: {conversation_response.status_code}")
    print(conversation_response.text)

Conversation fetched successfully!
{
  "name": "projects/llm-studies/locations/global/conversations/conversation_1",
  "agents": [
    "projects/llm-studies/locations/global/dataAgents/data_agent_1"
  ],
  "createTime": "2026-03-09T16:25:46.528536Z",
  "lastUsedTime": "2026-03-09T16:25:47.051255Z"
}


In [24]:
conversation_url = f"{base_url}/{api_version}/projects/{billing_project}/locations/{location}/conversations"


conversation_response = requests.get(conversation_url, headers=headers)

# Handle the response
if conversation_response.status_code == 200:
    print("Conversation fetched successfully!")
    print(json.dumps(conversation_response.json(), indent=2))
else:
    print(f"Error while fetching conversation: {conversation_response.status_code}")
    print(conversation_response.text)

Conversation fetched successfully!
{
  "conversations": [
    {
      "name": "projects/llm-studies/locations/global/conversations/conversation_1",
      "agents": [
        "projects/llm-studies/locations/global/dataAgents/data_agent_1"
      ],
      "createTime": "2026-03-09T16:25:46.528536Z",
      "lastUsedTime": "2026-03-09T16:25:47.051255Z"
    }
  ]
}


In [25]:
# @title Streaming Chat Messages

def is_json(str):
    try:
        json_object = json_lib.loads(str)
    except ValueError:
        return False
    return True

def handle_text_response(resp):
    parts = resp["parts"]
    full_text = "".join(parts)
    if "\n" not in full_text and len(full_text) > 80:
        wrapped_text = textwrap.fill(full_text, width=80)
        print(wrapped_text)
    else:
        print(full_text)

def get_property(data, field_name, default=""):
    return data[field_name] if field_name in data else default

def display_schema(data):
    fields = data["fields"]
    df = pd.DataFrame(
        {
            "Column": map(lambda field: get_property(field, "name"), fields),
            "Type": map(lambda field: get_property(field, "type"), fields),
            "Description": map(
                lambda field: get_property(field, "description", "-"), fields
            ),
            "Mode": map(lambda field: get_property(field, "mode"), fields),
        }
    )
    display(df)

def display_section_title(text):
    display(HTML(f"<h2>{text}</h2>"))

def format_bq_table_ref(table_ref):
    return "{}.{}.{}".format(
        table_ref["projectId"], table_ref["datasetId"], table_ref["tableId"]
    )

def format_looker_table_ref(table_ref):
    return "lookmlModel: {}, explore: {}".format(
        table_ref["lookmlModel"], table_ref["explore"]
    )

def display_datasource(datasource):
    source_name = ""

    if "studioDatasourceId" in datasource:
        source_name = datasource["studioDatasourceId"]
    elif "lookerExploreReference" in datasource:
        source_name = format_looker_table_ref(datasource["lookerExploreReference"])
    else:
        source_name = format_bq_table_ref(datasource["bigqueryTableReference"])

    print(source_name)
    if "schema" in datasource:
        display_schema(datasource["schema"])

def handle_schema_response(resp):
    if "query" in resp:
        print(resp["query"]["question"])
    elif "result" in resp:
        display_section_title("Schema resolved")
        print("Data sources:")
        for datasource in resp["result"]["datasources"]:
            display_datasource(datasource)

def handle_data_response(resp):
    if "query" in resp:
        query = resp["query"]
        display_section_title("Retrieval query")
        if "name" in query:
            print("Query name: {}".format(query["name"]))
        if "question" in query:
          print("Question: {}".format(query["question"]))
        if "datasources" in query:
          print("Data sources:")
          for datasource in query["datasources"]:
              display_datasource(datasource)
    elif "generatedSql" in resp:
        display_section_title("SQL generated")
        print(resp["generatedSql"])
    elif "result" in resp:
        display_section_title("Data retrieved")

        fields = map(
            lambda field: get_property(field, "name"),
            resp["result"]["schema"]["fields"],
        )
        dict = {}

        for field in fields:
            dict[field] = [get_property(el, field) for el in resp["result"]["data"]]

        display(pd.DataFrame(dict))

def handle_chart_response(resp):
    if "query" in resp:
        print(resp["query"]["instructions"])
    elif "result" in resp:
        vegaConfig = resp["result"]["vegaConfig"]
        alt.Chart.from_json(json_lib.dumps(vegaConfig)).display()

def handle_clarification_response(resp):
    display_section_title("Clarification required")

    clarification_questions = ""
    for q in resp["questions"]:
        # Add the question text
        clarification_questions += f"{q['question']}\n"
        # Add the options as bullet points
        for option in q["options"]:
            clarification_questions += f"* {option}\n"
        clarification_questions += "\n" # Add a newline between questions
    print(clarification_questions)

def handle_error(resp):
    display_section_title("Error")
    print("Code: {}".format(resp["code"]))
    print("Message: {}".format(resp["message"]))

def get_stream(url, json):
    s = requests.Session()

    acc = ""

    with s.post(url, json=json, headers=headers, stream=True) as resp:
        for line in resp.iter_lines():
            if not line:
                continue

            decoded_line = str(line, encoding="utf-8")

            if decoded_line == "[{":
                acc = "{"
            elif decoded_line == "}]":
                acc += "}"
            elif decoded_line == ",":
                continue
            else:
                acc += decoded_line

            if not is_json(acc):
                continue

            data_json = json_lib.loads(acc)

            if "systemMessage" not in data_json:
                if "error" in data_json:
                    handle_error(data_json["error"])
                continue

            if "text" in data_json["systemMessage"]:
                handle_text_response(data_json["systemMessage"]["text"])
            elif "schema" in data_json["systemMessage"]:
                handle_schema_response(data_json["systemMessage"]["schema"])
            elif "data" in data_json["systemMessage"]:
                handle_data_response(data_json["systemMessage"]["data"])
            elif "chart" in data_json["systemMessage"]:
                handle_chart_response(data_json["systemMessage"]["chart"])
            elif "clarification" in data_json["systemMessage"]:
                handle_clarification_response(data_json["systemMessage"]["clarification"])
            else:
                colored_json = highlight(
                    acc, lexers.JsonLexer(), formatters.TerminalFormatter()
                )
                print(colored_json)
            print("\n")
            acc = ""

In [26]:
data_agent_id = "data_agent_1"      # @param {type:"string"}
conversation_id = "conversation_1"  # @param {type:"string"}

# fmt: off
question = "Make a bar graph for the top 5 states by the total number of airports"  # @param {type:"string"}
# fmt: on

chat_url = (
    f"{base_url}/{api_version}/projects/{billing_project}/locations/{location}:chat"
)

 # Construct the payload
chat_payload = {
    "parent": f"projects/{billing_project}/locations/global",
    "messages": [{"userMessage": {"text": question}}],
    "conversation_reference": {
        "conversation": f"projects/{billing_project}/locations/{location}/conversations/{conversation_id}",
        "data_agent_context": {
            "data_agent": f"projects/{billing_project}/locations/{location}/dataAgents/{data_agent_id}",
        },
    },
}

if looker_selected:
    chat_payload["conversation_reference"]["data_agent_context"]["credentials"] = (
        looker_credentials
    )

# Call get_stream method to stream the response
get_stream(chat_url, chat_payload)

Retrieved contextRetrieved context for 2 tables.


Constructing the SQL QueryI've successfully identified the table and drafted the
SQL query. The key is now generating the Python code to visualize the resulting
data as a bar graph, using the `google:python_interpreter`. I am ready to move
to the next step.


Analyzing Airport DataI've zeroed in on the top 5 states by airport count. Now,
I am structuring the data. I'm building a plan to use Altair to make a bar chart
and identifying data names, chart IDs, and column details. Encoding will be
handled by specifying 'X' and 'Y' for the appropriate columns. The goal is to
produce the final answer, a bar chart of the data.


Considering Airport DataI've reviewed the data and the chart is ready to go. My
focus is on the top five states by airport count. Currently, I see that Texas is
leading by a large margin, followed by Florida, California, Alaska, and
Pennsylvania. I'm wondering why Texas has such a significant lead.


### Summary
This fi

Data sources:
bigquery-public-data.faa.us_airports




SELECT
    airports.state_abbreviation,
    COUNT(*) AS airport_count
FROM
    `bigquery-public-data.faa.us_airports` AS airports
GROUP BY
    airports.state_abbreviation
ORDER BY
    airport_count DESC
LIMIT 5;






,state_abbreviation,airport_count
0,TX,2033
1,FL,886
2,CA,870
3,AK,762
4,PA,759


Data sources:
bigquery-public-data.faa.us_airports




SELECT
    airports.state_abbreviation,
    COUNT(*) AS airport_count
FROM
    `bigquery-public-data.faa.us_airports` AS airports
GROUP BY
    airports.state_abbreviation
ORDER BY
    airport_count DESC
LIMIT 5;






KeyError: 'instructions'